# Installations

In [ ]:
!python -m pip install -U langchain-community langchain-huggingface tiktoken chromadb langchain langchain_ollama langchainhub sentence_transformers ollama keyboard SpeechRecognition pyttsx3 pyaudio 
!python -m pip install -U python-dotenv beautifulsoup4

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('config.env')
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL')
USER_AGENT = os.getenv('USER_AGENT')
VCPKG_PATH = os.getenv('VCPKG_PATH')

# Document Loading

Loading and parsing all required documents to files for later use.

In [ ]:
#imports, for testing or otherwise
import time
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

In [ ]:
#Test load, to see how data is formatted to move into the correct list
test_splitter = RecursiveCharacterTextSplitter(chunk_size=100, separators=["No. "], chunk_overlap=0,keep_separator=False)

test_link = "https://ffxivcollect.com/triad/cards"
test_name = re.sub('.*/','',test_link).strip()

print(f"Testing {test_link} page...")
test_loader = WebBaseLoader(test_link)
test_data = test_loader.load()
print(f"Link {test_link} loaded.")
test_split = test_splitter.split_documents(test_data)
with open(f'data/test/{test_name}_test.txt', 'w', encoding='utf-8') as f:
    for test_line in test_split:
        f.write(f"{str(test_line).replace("\n", "^")}\n")
print(test_split)

In [ ]:
import shutil
spreadsheet_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0,separators=["\n\n\n\n\n\n"],keep_separator=False)
all_splits=[]

#FFXIV Wiki pages
file_list = ["wiki_links.txt","collect_links.txt"]
link_list = open("wiki_links.txt", 'r')
timeout = 10000
db_name = "their_brain_kupo"
#Progress:
#ALL COLLECT 10/10
#Bone 7/10
#Cloth 5/10
#Ingredient 10/10
#Seafood; the perfect child, 10/10
#Metal; 7/10, mostly there and grouped but needs cleanup
#miner/botanist node locations, 11/10
split_selector = {'Airship_Components' : [100, ["\n\n\n\n\n\n", "\n\n\n\n"], False], 'bardings': [100, ["\n\n\n\n\n\n"], False], 'Bone': [100, ["\n\n\n\n\n\n", "\n\n\n"], False], 'Botanist_Node_Locations': [100, ["\n\n\n\n\n\n", "\n\n\n\n"], False], 'cards': [100, ["No. "], True], 'Cloth': [200, ["\n\n\n\n\n\n", "\n\n\n"], False] , 'Dye': [100, ["\n\n\n\n\n\n"], False] , 'emotes': [100, ["\n\n\n\n\n\n"], False], 'facewear': [100, ["\n\n\n\n\n\n"], False], 'frames': [100, ["\n\n\n\n\n\n"], False], 'Ingredient': [100, ["\n\n\n\n\n\n", "\n\n\n"], False] , 'Leather': [200, ["\n\n\n\n\n\n", "\n\n\n"], False] , 'Lumber': [100, ["\n\n\n\n\n\n"], False] ,'Metal' : [100, ["\n\n\n\n\n\n", "\n\n\n"], False], 'Miner_Node_Locations': [100, ["\n\n\n\n\n\n", "\n\n\n\n"], False], 'minions': [100, ["\n\n\n\n\n\n"], False], 'mounts': [100, ["\n\n\n\n\n\n"], False], 'orchestrions': [100, ["\n\n\n\n\n\n"], False], 'Part': [100, ["\n\n\n\n\n\n", "\n\n\n"], False] , 'Reagent': [100, ["\n\n\n\n\n\n", "\n\n\n"], False] , 'Seafood' : [100, ["\n\n\n\n\n\n"], False], 'spells': [100, ["No. "], True], 'Stone_(Material)': [100, ["\n\n\n\n\n\n"], False] , 'Submersible_Components': [100, ["\n\n\n\n\n\n","\n\n\n\n"], False]}
db_exists = os.path.isdir(db_name)
if (not db_exists) or (db_exists and (time.time()-os.path.getmtime(db_name) > timeout)) or True:
    if db_exists:
        #TODO: see if we can deallocate usage if running
        print('db exists')
        try:
            print(type(vectorstore))
        except NameError:
            print ("Removing outdated db...")
            shutil.rmtree(db_name)
    for file in file_list:
        link_list = open(file, 'r')
        source_name = re.sub('_links.txt','',file).strip()
        print(f"Loading \'{source_name}\' pages...")
        for link in link_list:
            #TODO: dont make calls super often if the page has been pulled recently, like ~ the past day.
            link = link.strip()
            name = re.sub('.*/','',link).strip()
            write_path = f'data/{source_name}/{name}.txt'
            file_exists = os.path.isfile(write_path)
            loader = WebBaseLoader(link)
            data = loader.load()
            print(f"Link {link} loaded.")
            this_splitter = RecursiveCharacterTextSplitter(chunk_size=split_selector[name][0], chunk_overlap=0,separators=split_selector[name][1],keep_separator=split_selector[name][2])
            link_split = this_splitter.split_documents(data)
            all_splits += link_split
            with open(write_path, 'w', encoding='utf-8') as f:
                for line in link_split:
                    f.write(f"{str(line).replace("\n", " ")}\n")
                f.close()

print(f"Total of data points created: {len(all_splits)}")

In [ ]:
print(f"Total of data points created: {len(all_splits)}")
db_exists = os.path.isfile("their_brain_kupo")
print(db_exists)
db_name = "their_brain_kupo"

In [ ]:
#from langchain_text_splitters import RecursiveCharacterTextSplitter
#
#text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0,separators=["\n\n\n\n\n\n"],keep_separator=False)
#all_splits = text_splitter.split_documents(data)
#print(all_splits)
#print(all_splits)

# Index

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

#embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#upgrading to this once I get a good connection...
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L12-v2")

In [ ]:
# Add to ChromaDB vector store
from langchain_community.vectorstores import Chroma
db_exists = os.path.isfile("their_brain_kupo.txt")
if not db_exists:
    vectorstore = Chroma.from_documents(
            documents=all_splits,
            collection_name="rag-chroma",
            embedding=embeddings,
            persist_directory=db_name
    )
else:
    vectorstore = Chroma(
        collection_name="rag-chroma",
        embedding=embeddings,
        persist_directory=db_name
    )
retriever = vectorstore.as_retriever(search_type='mmr', search_kwargs={"k": 20, "fetch_k":50})

In [ ]:
question = "What is spell No. 34"
docs = vectorstore.similarity_search(question, k=100)
len(docs)

In [ ]:
docs[0]

# Model

We'll be using gemma3n:e4b, to try and get something efficient/able to run on a laptop reasonably.

# Install Ollama

In [ ]:
#Probably best to just do the site install, but in case you need it done from the CLI
#!curl https://ollama.ai/install.sh | bash

In [ ]:
#add Ollama to PATH
os.environ.get("PATH")
ollama_path=r"c:\Users\Donal\AppData\Local\Programs\Ollama"
os.environ["PATH"] += os.pathsep + ollama_path
os.environ.get("PATH")

# Set up Ollama Server

In [ ]:
import subprocess
import time

# Start ollama as a backrgound process
command = "ollama serve&"

# Use subprocess.Popen to start the process in the background
process = subprocess.run(command,
                            shell=True,capture_output=True)
print("Process ID:", process)

In [ ]:
version = subprocess.run("ollama -v", shell=True, capture_output=True)
print(version)
!ollama -v

In [ ]:
!ollama pull gemma3n:e4b

In [31]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}

Format your response as if you are a Moogle, namely by ending some sentances (not all) with the phrase "Kupo!".

Do not reference the fact you are looking at documents. Please keep responses to at most 5 sentances.

If you are including a location/coordinates, limit your response to only include places mentioned in the Question. Not every response will require coordinates.

Percentages are most likely the acquired rate, as in that percent of players have obtained that thing.
"""
prompt = ChatPromptTemplate.from_template(template)

# Local LLM
ollama_llm = OLLAMA_MODEL
model_local = ChatOllama(model=ollama_llm, num_predict=512)

# Chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model_local
    | StrOutputParser()
)

In [32]:
# Question
import time
import keyboard
import speech_recognition as sr
import pyttsx3
from rich import print

r = sr.Recognizer()

print("[green]Starting the loop, press F4 to begin")
while True:
    # Wait until user presses "f4" key
    if keyboard.read_key() != "f4":
        time.sleep(0.1)
        continue

    try:
        with sr.Microphone() as source:
            print("[green]Listening...")
            r.adjust_for_ambient_noise(source, duration=0.2)
            audio = r.listen(source)
            text = r.recognize_google(audio)
            mic_result = text.lower()  
            print("You said:", mic_result)

    except sr.RequestError as e:
        print("Could not request results; {0}".format(e))

    except sr.UnknownValueError:
        print("Could not understand audio, press again")
        continue

    except KeyboardInterrupt:
        print("Program terminated by user")
        break

    # Get question from mic
    
    if mic_result == '':
        print("[red]Did not receive any input from your microphone!")
        continue
    elif "goodbye" in mic_result:
        print("You have bid the Kupo fairwell!")
        break
    print("[green]Asking Kupo...")
    response = chain.invoke(mic_result)
    #setting tts oneshot
    tts = pyttsx3.init()
    voices = tts.getProperty('voices')       # getting details of current voice
    tts.setProperty('voice', voices[1].id)   # changing index, changes voices. 1 for female
    print(response)
    tts.say(response)
    tts.runAndWait()
    tts.stop()
    del tts
    print("[green]Ready for next input!")

Starting the loop, press F4 to begin

Listening...

You said: where can i find a moo

Asking Kupo...

Oh my! A moo, you say? Well, there are many moogles to be found, Kupo! You can find a Fat Mooogle in the Online 
Store, Kupo! Or perhaps a Wind-up Mooogle from the A Realm Reborn Collector's Edition, Kupo! You might also find a 
Mooogle Mogpom near the Lakes in The Churning Mist, Kupo!

Ready for next input!

... (29 outputs left out)

Ready for next input!

Listening...

You said: goodbye

You have bid the Kupo fairwell!

In [33]:
tts.stop()
del tts

NameError: name 'tts' is not defined